# 02 - Vega and Curvature Paths

Synthetic engineering and validation evidence, not final regulatory capital.

Use this notebook with the [SBM package journey](../docs/PACKAGE_JOURNEY.md) and the executable [package demo](../examples/run_demo.py).

Raw inputs your upstream risk engine must emit: canonical SBM sensitivity rows, or one Arrow table per homogeneous `(risk_class, risk_measure)` path, with deterministic sensitivity identifiers, desk and legal-entity lineage, bucket labels, risk-factor vertices, and signed sensitivity amounts. The machine-readable GIRR delta input-table example is [`frtb_sbm.girr_delta.schema.json`](../../../docs/schemas/input_table/frtb_sbm.girr_delta.schema.json); the package dataset contract is [`DATASET_CONTRACT.md`](../docs/DATASET_CONTRACT.md).

This notebook replays the GIRR and non-GIRR vega fixtures, then builds a compact synthetic curvature portfolio to show branch evidence in the audit payload. Inputs are synthetic development examples and are not final regulatory capital outputs.


In [ ]:
from pathlib import Path
import sys

HERE = Path.cwd()
PACKAGE_ROOT = None
REPO_ROOT = None
for candidate in (HERE, *HERE.parents):
    if (candidate / "src" / "frtb_sbm").exists():
        PACKAGE_ROOT = candidate
        REPO_ROOT = candidate.parent.parent if candidate.parent.name == "packages" else candidate
        break
    if (candidate / "packages" / "frtb-sbm" / "src" / "frtb_sbm").exists():
        REPO_ROOT = candidate
        PACKAGE_ROOT = candidate / "packages" / "frtb-sbm"
        break
if PACKAGE_ROOT is None:
    raise RuntimeError("Could not locate frtb-sbm package root")
workspace_src_paths = tuple(sorted((REPO_ROOT / "packages").glob("*/src"))) if REPO_ROOT else ()
for path in (
    PACKAGE_ROOT / "examples",
    PACKAGE_ROOT,
    PACKAGE_ROOT / "src",
    *workspace_src_paths,
    REPO_ROOT,
):
    if path is not None and str(path) not in sys.path:
        sys.path.insert(0, str(path))
PACKAGE_ROOT

In [ ]:
try:
    from IPython.display import Markdown, display
except ImportError:

    class Markdown(str):
        pass

    def display(value: object) -> None:
        print(value)


from frtb_sbm import calculate_sbm_capital, serialize_sbm_result, validate_sbm_result_reconciliation
from sbm_notebook_data import (
    curvature_sample_sensitivities,
    load_fixture,
    markdown_table,
    notebook_context,
)

## Public API happy path

This notebook runs supported vega and curvature paths through the top-level row API: `calculate_sbm_capital`.


In [ ]:
vega_rows = []
vega_payloads = {}
for fixture_id in ("girr_vega_v1", "non_girr_vega_v1"):
    fixture = load_fixture(fixture_id)
    result = calculate_sbm_capital(fixture.sensitivities, context=fixture.context)
    validate_sbm_result_reconciliation(result)
    payload = serialize_sbm_result(result)
    vega_payloads[fixture_id] = payload
    vega_rows.append(
        (
            fixture_id,
            len(fixture.sensitivities),
            payload["total_capital"],
            fixture.expected_outputs["total_capital"],
            payload.get("selected_portfolio_scenario", ""),
        )
    )

display(
    Markdown(
        markdown_table(
            ("Fixture", "Input rows", "Total capital", "Expected total", "Portfolio scenario"),
            vega_rows,
        )
    )
)

## Implementation details / math verification - Vega and curvature branch evidence

The remaining cells inspect scenario totals, selected curvature branches, and audit payload evidence.


In [ ]:
non_girr_payload = vega_payloads["non_girr_vega_v1"]
scenario_totals = non_girr_payload.get("portfolio_scenario_totals", {})
display(
    Markdown(
        markdown_table(
            ("Portfolio scenario", "Total"),
            sorted(scenario_totals.items()),
        )
    )
)

In [ ]:
curvature_sensitivities = curvature_sample_sensitivities()
curvature_context = notebook_context("sbm-curvature-notebook")
curvature_result = calculate_sbm_capital(curvature_sensitivities, context=curvature_context)
validate_sbm_result_reconciliation(curvature_result)
curvature_payload = serialize_sbm_result(curvature_result)

display(
    Markdown(
        markdown_table(
            ("Rows", "Risk classes", "Total capital", "Selected portfolio scenario"),
            (
                (
                    len(curvature_sensitivities),
                    len(curvature_payload["risk_classes"]),
                    curvature_payload["total_capital"],
                    curvature_payload.get("selected_portfolio_scenario", ""),
                ),
            ),
        )
    )
)

In [ ]:
curvature_rows = []
for risk_class in curvature_payload["risk_classes"]:
    curvature_rows.append(
        (
            risk_class["risk_class"],
            risk_class.get("risk_measure", ""),
            risk_class["selected_capital"],
            len(risk_class.get("curvature_branches", [])),
            len(risk_class.get("curvature_bucket_branches", [])),
        )
    )

display(
    Markdown(
        markdown_table(
            ("Risk class", "Measure", "Selected capital", "Input branches", "Bucket branches"),
            curvature_rows,
        )
    )
)

In [ ]:
branch_rows = []
for risk_class in curvature_payload["risk_classes"]:
    for branch in risk_class.get("curvature_bucket_branches", []):
        branch_rows.append(
            (
                risk_class["risk_class"],
                branch["bucket_id"],
                branch["scenario"],
                branch["selected_branch"],
                branch["selected_bucket_capital"],
            )
        )

display(
    Markdown(
        markdown_table(
            ("Risk class", "Bucket", "Scenario", "Selected branch", "Selected bucket capital"),
            branch_rows[:12],
        )
    )
)

## See also

- [SBM package journey](../docs/PACKAGE_JOURNEY.md)
- [SBM dataset contract](../docs/DATASET_CONTRACT.md)
- [Client integration guide](../../../docs/CLIENT_INTEGRATION.md)
- [DRC package journey](../../frtb-drc/docs/PACKAGE_JOURNEY.md)
- [RRAO package journey](../../frtb-rrao/docs/PACKAGE_JOURNEY.md)
- [CVA package journey](../../frtb-cva/docs/PACKAGE_JOURNEY.md)
- [IMA package journey](../../frtb-ima/docs/PACKAGE_JOURNEY.md)
